# DEAP Experiment
EEG-based emotion classification using EEGNet on the DEAP dataset.
Includes preprocessing, training, and ablation study results.

In [ ]:
import os
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

In [ ]:
# Set path to DEAP data_preprocessed_python folder
data_path = "your/path/to/data_preprocessed_python"

## 1. EDA
Check signal range across subjects to determine normalization strategy.

In [ ]:
for i in [1, 10, 20, 32]:
    file_name = f's{i:02d}.dat'
    with open(os.path.join(data_path, file_name), 'rb') as f:
        subject = pickle.load(f, encoding='latin1')

    eeg = subject['data'][:, :32, 384:]
    print(f"s{i:02d} | min: {eeg.min():.2f} | max: {eeg.max():.2f} | "
          f"mean: {eeg.mean():.2f} | std: {eeg.std():.2f}")

## 2. Preprocessing
Note: DEAP preprocessed data already has a 4-45Hz bandpass filter applied.
Additional bandpass filtering was not applied to avoid redundant filtering.

In [ ]:
def preprocess_subject(file_path):
    with open(file_path, 'rb') as f:
        subject = pickle.load(f, encoding='latin1')

    # 1. Extract EEG 32 channels
    eeg = subject['data'][:, :32, :]

    # 2. Remove 3s baseline (384 points at 128Hz)
    eeg = eeg[:, :, 384:]

    # 3. Segment into 1s epochs
    eeg = eeg.reshape(40, 32, 60, 128)
    eeg = eeg.transpose(0, 2, 1, 3)
    eeg = eeg.reshape(-1, 32, 128)      # (2400, 32, 128)

    # 4. Subject-wise z-score normalization
    mean = eeg.mean()
    std = eeg.std() + 1e-6
    eeg = (eeg - mean) / std

    # 5. Add channel dimension for model input
    eeg = eeg[:, np.newaxis, :, :]      # (2400, 1, 32, 128)

    # 6. Binarize Valence labels using subject-wise median
    valence = subject['labels'][:, 0]
    valence_binary = (valence >= np.median(valence)).astype(int)
    labels = np.repeat(valence_binary, 60)

    return eeg, labels

In [ ]:
# Load all 32 subjects
all_eeg, all_labels = [], []

for i in range(1, 33):
    file_path = os.path.join(data_path, f's{i:02d}.dat')
    eeg, labels = preprocess_subject(file_path)
    all_eeg.append(eeg)
    all_labels.append(labels)

X = np.concatenate(all_eeg, axis=0)
y = np.concatenate(all_labels, axis=0)

print("Total data shape:", X.shape)
print("Total label shape:", y.shape)
print("Class distribution:", np.bincount(y))

## 3. Dataset and DataLoader

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # maintain class ratio
)

train_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train)),
    batch_size=64, shuffle=True
)
test_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_test), torch.LongTensor(y_test)),
    batch_size=64, shuffle=False
)

## 4. EEGNet Model

In [ ]:
class EEGNet_Block1(nn.Module):
    def __init__(self):
        super().__init__()
        self.temporal_conv = nn.Conv2d(1, 8, kernel_size=(1, 64), padding='same', bias=False)
        self.depthwise_conv = nn.Conv2d(8, 16, kernel_size=(32, 1), groups=8, bias=False)
        self.bn1 = nn.BatchNorm2d(8)
        self.bn2 = nn.BatchNorm2d(16)
        self.pool = nn.AvgPool2d(kernel_size=(1, 4))
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x):
        x = F.elu(self.bn1(self.temporal_conv(x)))
        x = F.elu(self.bn2(self.depthwise_conv(x)))
        return self.dropout(self.pool(x))


class EEGNet_Block2(nn.Module):
    def __init__(self):
        super().__init__()
        self.depthwise_conv = nn.Conv2d(16, 16, kernel_size=(1, 16), groups=16, padding='same', bias=False)
        self.pointwise_conv = nn.Conv2d(16, 16, kernel_size=(1, 1), bias=False)
        self.bn = nn.BatchNorm2d(16)
        self.pool = nn.AvgPool2d(kernel_size=(1, 8))
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x):
        x = F.elu(self.bn(self.pointwise_conv(self.depthwise_conv(x))))
        return self.dropout(self.pool(x))


class EEGNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.block1 = EEGNet_Block1()
        self.block2 = EEGNet_Block2()
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.block2(self.block1(x)))

## 5. Training

In [ ]:
model = EEGNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 100
best_acc = 0
patience = 20       # stop if no improvement for 20 epochs
counter = 0

for epoch in range(epochs):
    model.train()
    train_loss, correct, total = 0, 0, 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        correct += (output.argmax(dim=1) == y_batch).sum().item()
        total += y_batch.size(0)

    train_acc = correct / total * 100

    model.eval()
    test_correct, test_total = 0, 0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            output = model(X_batch)
            test_correct += (output.argmax(dim=1) == y_batch).sum().item()
            test_total += y_batch.size(0)

    test_acc = test_correct / test_total * 100

    if (epoch + 1) % 5 == 0:
        print(f"Epoch: {epoch+1:3d}/{epochs} | "
              f"Loss: {train_loss/len(train_loader):.4f} | "
              f"Train: {train_acc:.2f}% | "
              f"Test: {test_acc:.2f}%")

    if test_acc > best_acc:
        best_acc = test_acc
        counter = 0
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        counter += 1
        if counter >= patience:
            print(f"Early Stopping at Epoch {epoch+1}")
            break

print(f"Best test accuracy: {best_acc:.2f}%")